In [1]:
# ============================================================
# NB06_Scoring_and_Gold_Output (FINAL CLEAN VERSION)
# One gate section. One scoring section. No duplicates.
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F
from datetime import datetime
import uuid

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
SCREEN_DATE = datetime.utcnow().strftime("%Y-%m-%d")

print(f"RUN_ID: {RUN_ID}")
print(f"SCREEN_DATE: {SCREEN_DATE}")


# ============================================================
# 1. READ CONFIG
# ============================================================

params = (
    spark.read.format("csv").option("header", "true")
    .load("Files/config/model_parameters.csv")
    .toPandas().set_index("parameter")["value"].to_dict()
)

MIN_PRICE_DAYS  = int(params.get("min_price_history_days", 260))
MIN_TURNOVER    = float(params.get("min_avg_daily_turnover_inr", 10000000))
MIN_CR          = float(params.get("min_current_ratio", 1.5))
MAX_DE          = float(params.get("max_debt_to_equity", 1.0))
MIN_OCF_NI      = float(params.get("min_ocf_to_net_income", 0.8))
MIN_RS          = float(params.get("min_rs_rank", 70))
MAX_52W_DIST    = float(params.get("max_distance_from_52w_high", -0.15))
MIN_Q_EPS       = float(params.get("min_quarterly_eps_yoy", 0.20))
MIN_Q_SAL       = float(params.get("min_quarterly_sales_yoy", 0.20))
MAX_PER_SECTOR  = int(params.get("max_stocks_per_sector", 2))
TARGET_COUNT    = int(params.get("target_candidate_count", 15))

print("Parameters loaded:")
for k, v in params.items():
    print(f"  {k} = {v}")


# ============================================================
# 2. LOAD SILVER TABLES
# ============================================================

price_latest = spark.sql("""
    SELECT
        symbol, trade_date, close, sma_200, high_252,
        mom_12_1, return_12m, avg_volume_50, avg_turnover_20,
        distance_from_52w_high, above_sma_200, price_days,
        rs_rank, sufficient_history_flag
    FROM silver_price_features
    WHERE trade_date = (SELECT MAX(trade_date) FROM silver_price_features)
""")

fund = spark.read.table("silver_fundamental_features")

regime = spark.sql("""
    SELECT trade_date, close AS nifty_close,
           sma_200 AS nifty_sma_200, market_regime
    FROM silver_market_regime
    ORDER BY trade_date DESC LIMIT 1
""")

universe = spark.read.table("bronze_universe_snapshot")

print(f"Price rows: {price_latest.count()}")
print(f"Fundamental rows: {fund.count()}")


# ============================================================
# 3. JOIN ALL TABLES
# ============================================================

# Drop duplicate columns from fund before joining
fund_clean = fund.drop("yahoo_symbol", "company_name", "industry")

joined = (
    universe
    .filter(
        (F.col("instrument_type") == "EQUITY") &
        (F.col("eligible_for_stock_screen") == True)
    )
    .join(price_latest, on="symbol", how="left")
    .join(fund_clean, on="symbol", how="left")
)

regime_broadcast = regime.select("market_regime", "nifty_close", "nifty_sma_200")
joined = joined.crossJoin(regime_broadcast)

print(f"Joined rows: {joined.count()}")


# ============================================================
# 4. STAGE A — ALL 11 ELIGIBILITY GATES (One section, no duplicates)
# ============================================================

scored = joined

# --- Gate 1-3: Price & Data Quality ---
scored = scored.withColumn(
    "gate_price_history",
    (F.col("price_days").isNotNull()) & (F.col("price_days") >= MIN_PRICE_DAYS)
)
scored = scored.withColumn(
    "gate_liquidity",
    (F.col("avg_turnover_20").isNotNull()) & (F.col("avg_turnover_20") >= MIN_TURNOVER)
)
scored = scored.withColumn(
    "gate_fresh_financials",
    F.col("financial_fresh_flag") == True
)

# --- Gate 4-6: Profitability (Current Year) ---
scored = scored.withColumn(
    "gate_ebit_positive",
    (F.col("ebit").isNotNull()) & (F.col("ebit") > 0)
)
scored = scored.withColumn(
    "gate_ni_positive",
    (F.col("net_income_ann").isNotNull()) & (F.col("net_income_ann") > 0)
)
scored = scored.withColumn(
    "gate_ocf_positive",
    (F.col("operating_cash_flow_ann").isNotNull()) & (F.col("operating_cash_flow_ann") > 0)
)

# --- Gate 7-9: Balance Sheet Safety ---

# RELAXED: Current Ratio >= 1.0 (lowered from 1.5),
# with Debt/Equity fallback when current ratio is missing
MIN_CR_RELAXED = 1.0

scored = scored.withColumn(
    "gate_current_ratio",
    F.when(
        F.col("current_ratio").isNotNull(),
        F.col("current_ratio") >= MIN_CR_RELAXED   # Lowered from 1.5 to 1.0
    ).when(
        F.col("debt_to_equity").isNotNull() & (F.col("debt_to_equity") <= 1.0),
        F.lit(True)   # Fallback: if D/E is low, pass even if CR is unknown
    ).otherwise(F.lit(True))  # Pass if both missing (don't reject for missing data)
)

scored = scored.withColumn(
    "gate_debt_equity",
    (F.col("debt_to_equity").isNotNull()) & (F.col("debt_to_equity") <= MAX_DE)
)
scored = scored.withColumn(
    "gate_ocf_ni",
    (F.col("ocf_to_ni").isNotNull()) & (F.col("ocf_to_ni") >= MIN_OCF_NI)
)

# --- Gate 10: Graham 4-Year Earnings Stability (RELAXED: 2 of 4 years) ---
# At least 2 of 4 years must have positive Net Income
scored = scored.withColumn(
    "gate_4y_earnings_stability",
    F.when(
        F.col("positive_earnings_4y_count").isNotNull(),
        F.col("positive_earnings_4y_count") >= 2
    ).otherwise(F.lit(True))  # If data missing, pass (don't block)
)

# --- Gate 11: Lynch 4-Year Cash Backing (RELAXED: 50% cash backing) ---
# Cumulative 4-year OCF must be >= 50% of cumulative 4-year Net Income,
# falling back to the single-year ocf_to_ni ratio (already computed
# above in this file) when the 4-year figure isn't available.
scored = scored.withColumn(
    "gate_4y_cash_backing",
    F.when(
        F.col("cum_4yr_ocf_to_ni").isNotNull(),
        F.col("cum_4yr_ocf_to_ni") >= 0.50
    ).when(
        F.col("ocf_to_ni") >= 0.80,
        F.lit(True)
    ).otherwise(F.lit(True))  # If no data at all, pass (don't block)
)

# --- Combine All 11 Gates ---
all_gates = (
    F.col("gate_price_history") &
    F.col("gate_liquidity") &
    F.col("gate_fresh_financials") &
    F.col("gate_ebit_positive") &
    F.col("gate_ni_positive") &
    F.col("gate_ocf_positive") &
    F.col("gate_current_ratio") &
    F.col("gate_debt_equity") &
    F.col("gate_ocf_ni") &
    F.col("gate_4y_earnings_stability") &
    F.col("gate_4y_cash_backing")
)

scored = scored.withColumn("all_gates_pass", all_gates)

# --- Eligibility Status & Reason ---
scored = scored.withColumn(
    "eligibility_status",
    F.when(F.col("all_gates_pass"), "ELIGIBLE").otherwise("INELIGIBLE")
)

scored = scored.withColumn(
    "eligibility_reason",
    F.when(F.col("all_gates_pass"), F.lit("All gates passed"))
    .when(~F.col("gate_price_history"), F.lit("Insufficient price history"))
    .when(~F.col("gate_liquidity"), F.lit("Low liquidity"))
    .when(~F.col("gate_fresh_financials"), F.lit("Stale financials"))
    .when(~F.col("gate_ebit_positive"), F.lit("Negative/missing EBIT"))
    .when(~F.col("gate_ni_positive"), F.lit("Negative/missing Net Income"))
    .when(~F.col("gate_ocf_positive"), F.lit("Negative/missing OCF"))
    .when(~F.col("gate_current_ratio"), F.lit("Current ratio < 1.0 or high debt"))
    .when(~F.col("gate_debt_equity"), F.lit(f"Debt/Equity > {MAX_DE}"))
    .when(~F.col("gate_ocf_ni"), F.lit(f"OCF/NI < {MIN_OCF_NI}"))
    .when(~F.col("gate_4y_earnings_stability"), F.lit("Earnings negative in 2+ of 4 years"))
    .when(~F.col("gate_4y_cash_backing"), F.lit("4-yr OCF < 70% of 4-yr NI"))
    .otherwise(F.lit("Multiple gates failed"))
)

# --- Diagnostic Print ---
gate_counts = {
    "price_history": scored.filter(~F.col("gate_price_history")).count(),
    "liquidity": scored.filter(~F.col("gate_liquidity")).count(),
    "fresh_financials": scored.filter(~F.col("gate_fresh_financials")).count(),
    "ebit_positive": scored.filter(~F.col("gate_ebit_positive")).count(),
    "ni_positive": scored.filter(~F.col("gate_ni_positive")).count(),
    "ocf_positive": scored.filter(~F.col("gate_ocf_positive")).count(),
    "current_ratio": scored.filter(~F.col("gate_current_ratio")).count(),
    "debt_equity": scored.filter(~F.col("gate_debt_equity")).count(),
    "ocf_ni": scored.filter(~F.col("gate_ocf_ni")).count(),
    "4y_earnings": scored.filter(~F.col("gate_4y_earnings_stability")).count(),
    "4y_cash": scored.filter(~F.col("gate_4y_cash_backing")).count(),
}
print("\nGate failures:")
for g, c in gate_counts.items():
    print(f"  {g}: {c} stocks failed")
print(f"  Total eligible: {scored.filter(F.col('all_gates_pass')).count()}")


# ============================================================
# 5. STAGE B — FACTOR SCORES (One section, no duplicates)
# ============================================================

w_all = Window.partitionBy(F.lit(1))

# --- QUALITY SCORE (35% of composite) ---
# 60% ROIC rank + 40% 4-Year Cash Truth rank

w_roic = w_all.orderBy("roic")
scored = scored.withColumn(
    "roic_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("roic_valid") == True) &
        (F.col("roic").isNotNull()),
        F.percent_rank().over(w_roic) * 100
    ).otherwise(F.lit(None).cast("double"))
)

w_cash = w_all.orderBy("cum_4yr_ocf_to_ni")
scored = scored.withColumn(
    "cash_truth_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("cum_4yr_ocf_to_ni").isNotNull()),
        F.percent_rank().over(w_cash) * 100
    ).otherwise(F.lit(None).cast("double"))
)

scored = scored.withColumn(
    "quality_score",
    F.when(
        F.col("roic_rank").isNotNull() & F.col("cash_truth_rank").isNotNull(),
        0.60 * F.col("roic_rank") + 0.40 * F.col("cash_truth_rank")
    ).when(
        F.col("roic_rank").isNotNull(),
        F.col("roic_rank")
    ).when(
        F.col("cash_truth_rank").isNotNull(),
        F.col("cash_truth_rank")
    ).otherwise(F.lit(None).cast("double"))
)

# --- VALUE SCORE (35% of composite) ---
# 70% Earnings Yield rank + 30% Inverse P/B rank

w_ey = w_all.orderBy("earnings_yield")
scored = scored.withColumn(
    "ey_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("ey_valid") == True) &
        (F.col("earnings_yield").isNotNull()),
        F.percent_rank().over(w_ey) * 100
    ).otherwise(F.lit(None).cast("double"))
)

# inv_pb_rank — sector-relative version.
#
# Requested change was:
#   OLD (absolute rank):
#     .orderBy(F.col("pb").desc())            -> percent_rank * 100
#   NEW (sector-relative):
#     .orderBy(F.col("pb_sector_zscore").asc()) -> percent_rank * 100
#
# NOTE ON DIRECTION: with percent_rank(), a window ordered *descending*
# puts the largest values first (rank ~0) and the smallest values last
# (rank ~100) — that's how the OLD line makes "inverse P/B" work: high
# pb -> low rank, low/cheap pb -> high rank (100 = cheapest = best).
# The NEW line as given orders pb_sector_zscore *ascending*, which is
# the opposite direction: it would put the cheapest-vs-sector stocks
# (most negative zscore) at rank ~0 and the most expensive-vs-sector
# stocks at rank ~100 — i.e. this would score the most overvalued
# stock highest, inverting the "inverse P/B" semantics used everywhere
# else in this file. Implemented exactly as requested (.asc()) below;
# if that inversion wasn't intended, swap .asc() for .desc() to match
# the original's cheap-is-good direction.
w_pb = w_all.orderBy(F.col("pb_sector_zscore").asc())
scored = scored.withColumn(
    "inv_pb_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("pb_sector_zscore").isNotNull()),
        F.percent_rank().over(w_pb) * 100
    ).otherwise(F.lit(None).cast("double"))
)

scored = scored.withColumn(
    "value_score",
    F.when(
        F.col("ey_rank").isNotNull() & F.col("inv_pb_rank").isNotNull(),
        0.70 * F.col("ey_rank") + 0.30 * F.col("inv_pb_rank")
    ).when(
        F.col("ey_rank").isNotNull(),
        F.col("ey_rank")
    ).when(
        F.col("inv_pb_rank").isNotNull(),
        F.col("inv_pb_rank")
    ).otherwise(F.lit(None).cast("double"))
)

# --- GROWTH SCORE (30% of composite) ---
# 50% 4-Year EPS CAGR rank + 30% Quarterly EPS YoY rank + 20% Quarterly Sales YoY rank

w_cagr4 = w_all.orderBy("eps_cagr_4y")
scored = scored.withColumn(
    "cagr_4y_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("eps_cagr_4y").isNotNull()),
        F.percent_rank().over(w_cagr4) * 100
    ).otherwise(F.lit(None).cast("double"))
)

w_qeps = w_all.orderBy("quarterly_eps_yoy")
scored = scored.withColumn(
    "q_eps_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("quarterly_eps_yoy").isNotNull()),
        F.percent_rank().over(w_qeps) * 100
    ).otherwise(F.lit(None).cast("double"))
)

w_qsal = w_all.orderBy("quarterly_sales_yoy")
scored = scored.withColumn(
    "q_sal_rank",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("quarterly_sales_yoy").isNotNull()),
        F.percent_rank().over(w_qsal) * 100
    ).otherwise(F.lit(None).cast("double"))
)

scored = scored.withColumn(
    "growth_score",
    F.when(
        F.col("cagr_4y_rank").isNotNull() &
        F.col("q_eps_rank").isNotNull() &
        F.col("q_sal_rank").isNotNull(),
        0.50 * F.col("cagr_4y_rank") +
        0.30 * F.col("q_eps_rank") +
        0.20 * F.col("q_sal_rank")
    ).when(
        F.col("cagr_4y_rank").isNotNull(),
        F.col("cagr_4y_rank")
    ).when(
        F.col("q_eps_rank").isNotNull(),
        F.col("q_eps_rank")
    ).otherwise(F.lit(None).cast("double"))
)

# --- COMPOSITE SCORE (35% Q + 35% V + 30% G) ---
scored = scored.withColumn(
    "composite_score",
    F.when(
        (F.col("all_gates_pass") == True) &
        F.col("quality_score").isNotNull() &
        F.col("value_score").isNotNull() &
        F.col("growth_score").isNotNull(),
        F.round(
            0.35 * F.col("quality_score") +
            0.35 * F.col("value_score") +
            0.30 * F.col("growth_score"),
            2
        )
    ).otherwise(F.lit(None).cast("double"))
)


# ============================================================
# 6. STAGE C — CONFIRMATION FLAGS (Do NOT change score)
# ============================================================

# Growth Confirmation (O'Neil: C, N, L, M)
scored = scored.withColumn(
    "growth_confirmation_flag",
    F.when(
        (F.col("all_gates_pass") == True) &
        (F.col("rs_rank") >= MIN_RS) &
        (F.col("distance_from_52w_high") >= MAX_52W_DIST) &
        (F.col("above_sma_200") == True) &
        (F.col("market_regime") == "RISK_ON") &
        (F.col("quarterly_eps_yoy") >= MIN_Q_EPS) &
        (F.col("quarterly_sales_yoy") >= MIN_Q_SAL),
        F.lit(True)
    ).otherwise(F.lit(False))
)

# Graham Value Flag (P/E x P/B <= 22.5)
scored = scored.withColumn(
    "graham_value_flag",
    F.when(
        (F.col("pe_proxy").isNotNull()) &
        (F.col("pb").isNotNull()) &
        (F.col("pe_proxy") > 0) &
        (F.col("pb") > 0) &
        (F.col("pe_proxy") * F.col("pb") <= 22.5),
        F.lit(True)
    ).otherwise(F.lit(False))
)


# ============================================================
# 7. STAGE D — 4-YEAR TREND FLAGS (Labels for dashboard)
# ============================================================

# Annual EPS YoY for each year
scored = scored.withColumn(
    "eps_yoy_1",
    F.when(
        (F.col("eps_y0").isNotNull()) & (F.col("eps_y1").isNotNull()) &
        (F.col("eps_y1") > 0) & (F.col("eps_y0") > 0),
        (F.col("eps_y0") / F.col("eps_y1") - 1) * 100
    ).otherwise(F.lit(None).cast("double"))
)

scored = scored.withColumn(
    "eps_yoy_2",
    F.when(
        (F.col("eps_y1").isNotNull()) & (F.col("eps_y2").isNotNull()) &
        (F.col("eps_y2") > 0) & (F.col("eps_y1") > 0),
        (F.col("eps_y1") / F.col("eps_y2") - 1) * 100
    ).otherwise(F.lit(None).cast("double"))
)

scored = scored.withColumn(
    "eps_yoy_3",
    F.when(
        (F.col("eps_y2").isNotNull()) & (F.col("eps_y3").isNotNull()) &
        (F.col("eps_y3") > 0) & (F.col("eps_y2") > 0),
        (F.col("eps_y2") / F.col("eps_y3") - 1) * 100
    ).otherwise(F.lit(None).cast("double"))
)

# EPS Accelerating / Decelerating
scored = scored.withColumn(
    "eps_accelerating",
    F.when(
        (F.col("eps_yoy_1").isNotNull()) &
        (F.col("eps_yoy_2").isNotNull()) &
        (F.col("eps_yoy_3").isNotNull()),
        (F.col("eps_yoy_1") > F.col("eps_yoy_2")) &
        (F.col("eps_yoy_2") > F.col("eps_yoy_3"))
    ).otherwise(F.lit(None).cast("boolean"))
)

scored = scored.withColumn(
    "eps_decelerating",
    F.when(
        (F.col("eps_yoy_1").isNotNull()) &
        (F.col("eps_yoy_2").isNotNull()) &
        (F.col("eps_yoy_3").isNotNull()),
        (F.col("eps_yoy_1") < F.col("eps_yoy_2")) &
        (F.col("eps_yoy_2") < F.col("eps_yoy_3"))
    ).otherwise(F.lit(None).cast("boolean"))
)

# OCF Improving (latest > oldest)
scored = scored.withColumn(
    "ocf_improving",
    F.when(
        (F.col("ocf_y0").isNotNull()) & (F.col("ocf_y3").isNotNull()),
        F.col("ocf_y0") > F.col("ocf_y3")
    ).otherwise(F.lit(None).cast("boolean"))
)

# Debt Growing / Free
scored = scored.withColumn(
    "debt_growing",
    F.when(
        (F.col("debt_y0").isNotNull()) & (F.col("debt_y3").isNotNull()),
        F.col("debt_y0") > F.col("debt_y3")
    ).otherwise(F.lit(None).cast("boolean"))
)

scored = scored.withColumn(
    "debt_free",
    F.when(
        (F.col("debt_y0").isNotNull()) & (F.col("debt_y0") <= 0),
        F.lit(True)
    ).otherwise(F.lit(False))
)

# Profit Quality (4-year OCF/NI)
scored = scored.withColumn(
    "profit_quality",
    F.when(F.col("cum_4yr_ocf_to_ni") >= 1.0, F.lit("HIGH"))
    .when(F.col("cum_4yr_ocf_to_ni") >= 0.8, F.lit("OK"))
    .when(F.col("cum_4yr_ocf_to_ni").isNotNull(), F.lit("LOW"))
    .otherwise(F.lit("N/A"))
)

# Graham Stable Earnings (3+ of 4 years positive)
scored = scored.withColumn(
    "graham_stable_earnings",
    F.when(
        (F.col("positive_earnings_4y_count").isNotNull()) &
        (F.col("positive_earnings_4y_count") >= 3),
        F.lit(True)
    ).otherwise(F.lit(False))
)

# Quarterly YoY display (rounded)
scored = scored.withColumn(
    "qtr_eps_yoy_display",
    F.round(F.col("qtr_eps_yoy_pct"), 1)
)
scored = scored.withColumn(
    "qtr_rev_yoy_display",
    F.round(F.col("qtr_rev_yoy_pct"), 1)
)


# ============================================================
# 8. WRITE gold_screen_current
# ============================================================

scored = scored.withColumn("screen_date", F.lit(SCREEN_DATE))
scored = scored.withColumn("calculation_timestamp", F.lit(RUN_TIMESTAMP))
scored = scored.withColumn("run_id", F.lit(RUN_ID))

gold_screen_cols = [
    # Identity
    "symbol", "yahoo_symbol", "company_name", "sector", "industry",
    "close", "market_cap",
    # Core ratios
    "roic", "earnings_yield", "current_ratio", "debt_to_equity",
    "ocf_to_ni", "pe_proxy", "pb", "peg",
    "eps_cagr_3y", "quarterly_eps_yoy", "quarterly_sales_yoy",
    # Price features
    "rs_rank", "distance_from_52w_high", "above_sma_200",
    "avg_turnover_20", "price_days",
    # Market regime
    "market_regime", "nifty_close", "nifty_sma_200",
    # Scores
    "quality_score", "value_score", "growth_score", "composite_score",
    # Flags
    "graham_value_flag", "growth_confirmation_flag",
    "eligibility_status", "eligibility_reason",
    # 4-Year Annual Data
    "eps_y0", "eps_y1", "eps_y2", "eps_y3",
    "eps_cagr_4y",
    "eps_yoy_1", "eps_yoy_2", "eps_yoy_3",
    "debt_y0", "debt_y3", "is_deleveraging",
    "rev_y0", "rev_y3", "rev_cagr_4y",
    "ocf_y0", "ocf_y3", "ocf_cagr_4y",
    "cum_4yr_ocf", "cum_4yr_ni", "cum_4yr_ocf_to_ni",
    "positive_earnings_4y_count",
    # Quarterly YoY
    "eps_q0", "eps_q4", "qtr_eps_yoy_pct",
    "rev_q0", "rev_q4", "qtr_rev_yoy_pct",
    "qtr_eps_yoy_display", "qtr_rev_yoy_display",
    # Trend Flags
    "eps_accelerating", "eps_decelerating",
    "ocf_improving", "debt_growing", "debt_free",
    "profit_quality", "graham_stable_earnings",
    # Audit
    "screen_date", "calculation_timestamp", "run_id"
]

# Only select columns that actually exist (safety)
existing_cols = [c for c in gold_screen_cols if c in scored.columns]
missing_cols = [c for c in gold_screen_cols if c not in scored.columns]
if missing_cols:
    print(f"WARNING: Columns not in scored: {missing_cols}")

gold_screen = scored.select(*existing_cols)

(
    gold_screen.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_screen_current")
)
print(f"✅ Saved: gold_screen_current ({gold_screen.count()} rows)")


# ============================================================
# 9. WRITE gold_stock_candidates (Top 15, sector cap)
# ============================================================

eligible_scored = scored.filter(
    (F.col("all_gates_pass") == True) &
    (F.col("composite_score").isNotNull())
)

print(f"Eligible & scored: {eligible_scored.count()}")

w_sector = Window.partitionBy("sector").orderBy(F.col("composite_score").desc())
w_global = Window.orderBy(F.col("composite_score").desc())

candidates = (
    eligible_scored
    .withColumn("sector_rank", F.row_number().over(w_sector))
    .filter(F.col("sector_rank") <= MAX_PER_SECTOR)
    .withColumn("candidate_rank", F.row_number().over(w_global))
    .filter(F.col("candidate_rank") <= TARGET_COUNT)
    .withColumn(
        "research_status",
        F.when(
            (F.col("eps_accelerating") == True) &
            (F.col("ocf_improving") == True),
            F.lit("GROWTH_ACCELERATING")
        ).when(
            (F.col("cum_4yr_ocf_to_ni") >= 1.0) &
            (F.col("debt_free") == True),
            F.lit("CASH_COMPOUNDER")
        ).when(
            (F.col("growth_confirmation_flag") == True),
            F.lit("GROWTH_CONFIRMED")
        ).when(
            (F.col("graham_value_flag") == True),
            F.lit("CORE_VALUE")
        ).otherwise(
            F.lit("CORE_CANDIDATE")
        )
    )
)

# Select specific columns for candidates
candidate_cols = [
    "candidate_rank", "symbol", "yahoo_symbol", "company_name",
    "sector", "industry", "close", "market_cap",
    "composite_score", "quality_score", "value_score", "growth_score",
    "roic", "earnings_yield", "pe_proxy", "pb", "peg",
    "eps_cagr_3y", "eps_cagr_4y",
    "quarterly_eps_yoy", "quarterly_sales_yoy",
    "rs_rank", "distance_from_52w_high",
    "growth_confirmation_flag", "graham_value_flag",
    "research_status",
    # 4-Year Trend
    "eps_y0", "eps_y1", "eps_y2", "eps_y3",
    "eps_yoy_1", "eps_yoy_2", "eps_yoy_3",
    "debt_y0", "debt_y3", "is_deleveraging",
    "ocf_y0", "ocf_y3", "ocf_cagr_4y",
    "cum_4yr_ocf_to_ni", "positive_earnings_4y_count",
    # Quarterly
    "qtr_eps_yoy_display", "qtr_rev_yoy_display",
    # Trend Flags
    "eps_accelerating", "eps_decelerating",
    "ocf_improving", "debt_growing", "debt_free",
    "profit_quality", "graham_stable_earnings",
    # Audit
    "screen_date", "run_id"
]

existing_cand_cols = [c for c in candidate_cols if c in candidates.columns]
missing_cand_cols = [c for c in candidate_cols if c not in candidates.columns]
if missing_cand_cols:
    print(f"WARNING: Candidate columns not in data: {missing_cand_cols}")

gold_candidates = candidates.select(*existing_cand_cols)

(
    gold_candidates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_stock_candidates")
)
print(f"✅ Saved: gold_stock_candidates ({gold_candidates.count()} rows)")


# ============================================================
# 10. APPEND to gold_score_history
# ============================================================

history_cols = [
    "symbol", "screen_date", "composite_score",
    "quality_score", "value_score", "growth_score",
    "eligibility_status", "growth_confirmation_flag",
    "graham_value_flag", "close", "run_id"
]

existing_hist_cols = [c for c in history_cols if c in scored.columns]

score_history = (
    scored
    .filter(F.col("composite_score").isNotNull())
    .select(*existing_hist_cols)
    .dropDuplicates(["symbol", "screen_date"])
)

(
    score_history.write
    .format("delta")
    .mode("append")
    .saveAsTable("gold_score_history")
)
print(f"✅ Appended: gold_score_history ({score_history.count()} rows)")

print("\n" + "=" * 55)
print(f"NB06 COMPLETE")
print(f"  Eligible: {scored.filter(F.col('all_gates_pass')).count()}")
print(f"  Candidates: {gold_candidates.count()}")
print(f"  History rows: {score_history.count()}")
print("=" * 55)

StatementMeta(, d5982c67-2526-48fb-9c57-124836d97d75, 3, Finished, Available, Finished, False)

RUN_ID: 192653a4-5678-41e6-9c4f-58f4692786a9
SCREEN_DATE: 2026-09-20
Parameters loaded:
  history_start_date = 1/1/2019
  min_price_history_days = 260
  min_avg_daily_turnover_inr = 10000000
  min_current_ratio = 1.5
  max_debt_to_equity = 1
  min_ocf_to_net_income = 0.8
  min_eps_cagr_3y = 0
  min_rs_rank = 70
  max_distance_from_52w_high = -0.15
  min_quarterly_eps_yoy = 0.2
  min_quarterly_sales_yoy = 0.2
  max_stocks_per_sector = 2
  target_candidate_count = 15
Price rows: 397
Fundamental rows: 398
Joined rows: 398

Gate failures:
  price_history: 10 stocks failed
  liquidity: 1 stocks failed
  fresh_financials: 1 stocks failed
  ebit_positive: 21 stocks failed
  ni_positive: 24 stocks failed
  ocf_positive: 35 stocks failed
  current_ratio: 0 stocks failed
  debt_equity: 51 stocks failed
  ocf_ni: 114 stocks failed
  4y_earnings: 20 stocks failed
  4y_cash: 48 stocks failed
  Total eligible: 240
✅ Saved: gold_screen_current (398 rows)
Eligible & scored: 240
✅ Saved: gold_stock_can

In [3]:
print(fund.columns)

StatementMeta(, 3c208aa5-da6e-45da-9e7a-88c8a120dd71, 5, Finished, Available, Finished, False)

['symbol', 'yahoo_symbol', 'company_name', 'sector', 'industry', 'annual_period_end', 'quarterly_period_end', 'market_cap', 'shares_outstanding', 'total_current_assets', 'total_current_liabilities', 'working_capital', 'net_ppe', 'capital_employed', 'total_debt', 'cash_and_equivalents', 'enterprise_value', 'total_equity', 'ebit', 'total_revenue_ann', 'net_income_ann', 'operating_cash_flow_ann', 'diluted_eps_ann', 'diluted_eps_ann_lag1', 'diluted_eps_ann_lag2', 'diluted_eps_ann_lag3', 'diluted_eps_qtr', 'diluted_eps_qtr_lag4', 'total_revenue_qtr', 'total_revenue_qtr_lag4', 'roic', 'earnings_yield', 'current_ratio', 'debt_to_equity', 'ocf_to_ni', 'quarterly_eps_yoy', 'quarterly_sales_yoy', 'eps_cagr_3y', 'pe_proxy', 'pb', 'peg', 'financial_fresh_flag', 'roic_valid', 'ey_valid', 'fundamentals_valid', 'eps_y0', 'eps_y1', 'eps_y2', 'eps_y3', 'eps_cagr_4y', 'debt_y0', 'debt_y3', 'is_deleveraging', 'rev_y0', 'rev_y3', 'rev_cagr_4y', 'ocf_y0', 'ocf_y3', 'ocf_cagr_4y', 'cum_4yr_ocf', 'cum_4yr_ni